# SCP Step 1 - Setup Tune Directory

This notebook prepares a tune directory so it is ready for the main SCP pipeline (Steps 2-6).

What Step 1 can do:
- download/cache Allen Database (ADB) model bundle files (`manifest.json`, morphology, fit json, `modfiles/`),
- compile modfiles (`nrnivmodl`) and optionally load the compiled mechanisms,
- scaffold common configs under `cell_configs/`,
- run validation checks (`manifest`, `load_cell`, `inputs.check_inputs`).

ADB setup is the primary example right now. Later generic/non-ADB setup should use the same tune-directory contract.

Script equivalent:
- `python scripts/step1_prepare.py --cell PV --tune seg_tuned --specimen-id 484635029`


In [ ]:
import json
import os
import sys
from pathlib import Path

# Lightweight bootstrap for kernels launched outside repo root.
try:
    from modules.notebooks.helpers import ensure_scp_repo_on_syspath
except ModuleNotFoundError:
    start = Path.cwd().resolve()
    injected = False
    for cand in (start, *start.parents):
        if (cand / "modules").is_dir() and (cand / "run_pipeline.py").is_file():
            if str(cand) not in sys.path:
                sys.path.insert(0, str(cand))
            injected = True
            break

    if not injected:
        for base in (start, start.parent):
            try:
                for child in base.iterdir():
                    if child.is_dir() and (child / "modules").is_dir() and (child / "run_pipeline.py").is_file():
                        if str(child) not in sys.path:
                            sys.path.insert(0, str(child))
                        injected = True
                        break
                if injected:
                    break
            except Exception:
                pass

    from modules.notebooks.helpers import ensure_scp_repo_on_syspath

repo_root = ensure_scp_repo_on_syspath()

print('SCP root:', repo_root)


## 1.1 User Settings

Choose the target tune directory and which setup actions to run.


In [ ]:
# Target tune location
cell_name = 'PN'               # e.g., PV, SST, PN
tune_name = 'adb_all'           # folder under cells/<cell>/tunes/
tunes_parent = 'tunes'
tune_dir_override = None        # optional absolute/relative path; overrides cell/tune settings

# ADB model identity
specimen_id = 485466109         # SST: 485466109, PV: 484635029; set None to use known-cell default
model_type = 'all active'       # e.g., perisomatic, all active

# Cell metadata scaffold (set None to use known-cell defaults)
soma_diam_multiplier = 1.0
cell_color = 'magenta'          # plotting color; e.g., 'magenta', 'cyan', 'orange'

# Optional ADB preview
LIST_AVAILABLE_ADB_MODELS = False

# Step-1 actions
DO_DOWNLOAD_ADB_BUNDLE = True
FORCE_DOWNLOAD = False
CACHE_STIMULUS = False

DO_COMPILE_MODFILES = True
RECOMPILE_MODFILES = False
LOAD_COMPILED_DLL = True
SORT_GENOME_ENTRIES_BY_SECTION = False
COERCE_GENOME_VALUES_TO_NUMERIC = False  # Converts numeric-like genome strings to floats (useful for all-active)

DO_SCAFFOLD_CONFIGS = True
CONFIG_MODE = 'fill'            # 'fill' | 'overwrite' | 'skip'
SYNC_CELL_METADATA = True

DO_VALIDATE = True
VALIDATE_INPUT_CONFIGS = True


In [ ]:
from modules.setup.adb import list_ADB_models
from modules.setup.step1_prepare import (
    guess_cell_color,
    guess_soma_multiplier,
    guess_specimen_from_cell,
)

if tune_dir_override:
    tune_dir = Path(tune_dir_override).expanduser().resolve()
else:
    tune_dir = (repo_root / 'cells' / cell_name / tunes_parent / tune_name).resolve()

if specimen_id is None:
    specimen_id = guess_specimen_from_cell(cell_name)
if specimen_id is None:
    raise ValueError('specimen_id is required for unknown cell labels; set it explicitly.')

if soma_diam_multiplier is None:
    soma_diam_multiplier = guess_soma_multiplier(cell_name)

if cell_color is None:
    cell_color = guess_cell_color(cell_name)

print('Tune dir:', tune_dir)
print('Cell:', cell_name, '| Tune:', tune_name)
print('Specimen:', specimen_id, '| Model type:', model_type)
print('Soma diam multiplier:', soma_diam_multiplier, '| Color:', cell_color)

if LIST_AVAILABLE_ADB_MODELS:
    list_ADB_models(specimen_id, filter_type=model_type)


In [ ]:
from modules.setup.step1_prepare import prepare_tune

summary = prepare_tune(
    tune_dir=tune_dir,
    cell_name=cell_name,
    tune_name=tune_name,
    specimen_id=int(specimen_id),
    model_type=model_type,
    soma_diam_multiplier=float(soma_diam_multiplier),
    color=cell_color,
    do_download=DO_DOWNLOAD_ADB_BUNDLE,
    force_download=FORCE_DOWNLOAD,
    cache_stimulus=CACHE_STIMULUS,
    do_compile_modfiles=DO_COMPILE_MODFILES,
    recompile_modfiles=RECOMPILE_MODFILES,
    load_compiled_dll=LOAD_COMPILED_DLL,
    coerce_genome_values_to_numeric=COERCE_GENOME_VALUES_TO_NUMERIC,
    sort_genome_entries_by_section=SORT_GENOME_ENTRIES_BY_SECTION,
    do_scaffold_configs=DO_SCAFFOLD_CONFIGS,
    config_mode=CONFIG_MODE,
    sync_cell_metadata=SYNC_CELL_METADATA,
    do_validate=DO_VALIDATE,
    validate_inputs_cfg=VALIDATE_INPUT_CONFIGS,
)

print(json.dumps(summary, indent=2))


## 1.2 Quick Check

The paths below should exist after a successful Step 1 setup run.


In [ ]:
check_paths = [
    tune_dir / 'manifest.json',
    tune_dir / 'modfiles',
    tune_dir / 'cell_configs' / 'cell_config.json',
    tune_dir / 'cell_configs' / 'sim_config.json',
    tune_dir / 'cell_configs' / 'geometry.json',
    tune_dir / 'cell_configs' / 'syn_config.json',
    tune_dir / 'cell_configs' / 'syn_groups' / 'placeholder_off.json',
]

for p in check_paths:
    print(f"{'OK' if p.exists() else 'MISSING'}  {p}")
